In [5]:
import tensorflow as tf
import numpy as np
from tensorflow.keras.models import Sequential 
from tensorflow.keras.layers import Dense, Input, Conv2D, MaxPooling2D, Flatten, GlobalAveragePooling2D
from sklearn.model_selection import train_test_split
from tensorflow.keras import layers, models
%pip install tqdm
from tqdm import tqdm
#from sklearn.metrics import confusion_matrix
#from sklearn.model_selection import training

Note: you may need to restart the kernel to use updated packages.


In [19]:
# crop image function

def crop_image(img):
    '''crops image into 300 x 300'''
    crop_box = (75, 75, 375, 375)
    cropped_img = img.crop(crop_box)
    cropped_img = img.resize((32,32)) #resize to 32 x 32
    cropped_img = img.convert('L') #returns a black and white image
    return cropped_img

In [20]:
# array assignment function

def array_assignment(im):
    '''breaks image down to four dimensional input array and assigns ground truth label array'''
    (width, height) = im.size

    for x_pixel in range(width):
        for y_pixel in range(height):
            #(R,G,B) = im.getpixel((x_pixel,y_pixel))

            # for dimension 4, 0=R, 1=G, 2=B
            #x[image_index,x_pixel,y_pixel,0] = R
            #x[image_index,x_pixel,y_pixel,1] = G
            #x[image_index,x_pixel,y_pixel,2] = B

            x[image_index, x_pixel, y_pixel] = im.getpixel((x_pixel,y_pixel))

    # assigns 0 to image if it does not have leukemia
    if "hem" in str(file_path):
        y[image_index] = 0

    # assigns 1 to image if it has leukemia
    elif "all" in str(file_path):
        y[image_index] = 1

    else: 
        pass
            

In [21]:
from pathlib import Path
from PIL import Image

print("import")

folder_path = Path("/Users/chloezeng/Documents/test png") #accesses folder with all our png images
png_files = folder_path.glob("*.png") #accesses individual file paths within our folder

print("files")

#x = np.empty((10661,300,300,3), dtype=np.uint8) #defines empty x (input) array
x = np.empty((10661,32,32), dtype=np.uint8) #FOR BLACK AND WHITE
y = np.empty((10661,), dtype=np.uint8) #defines empty y (ground truth/output) array

print("empty")

image_index = 0 #image number

for file_path in tqdm(png_files, desc="image processing progress:"):
    with Image.open(file_path) as img:
    #opens all images sequentially
        
        cropped_image = crop_image(img) #crops out extra black background
        
        array_assignment(cropped_image) #extracts data from pictures

        image_index += 1

print("crop")

#splits data into training, validation, and testing sets 80-10-10
x_train, x_temp, y_train, y_temp = train_test_split(x, y, test_size = 0.2, random_state = 30)
x_val, x_test, y_val, y_test = train_test_split(x_temp, y_temp, test_size = 0.5, random_state = 67) 

print("train test split")

print(x_train.shape)
print(x_val.shape)
print(x_test.shape)
print(y_train.shape)
print(y_val.shape)
print(y_test.shape)

import
files
empty
crop
train test split
(8528, 300, 300, 3)
(1066, 300, 300, 3)
(1067, 300, 300, 3)
(8528,)
(1066,)
(1067,)


In [22]:
#testing if the code actually works


from pathlib import Path

print("Current folder:", Path.cwd())

folder_path = Path("/Users/chloezeng/Documents/test png") #need to change this folder in a bit!
print("Folder exists:", folder_path.exists())

png_files = list(folder_path.glob("*.png"))
print("Number of PNG files:", len(png_files))

Current folder: /Users/chloezeng/leukemia-cnn-project-spis-2026-chloe-shandi
Folder exists: True
Number of PNG files: 10661


In [ ]:
# defining the CNN model

model = Sequential()

model.add(Input(shape=(300,300,3)))
model.add(Conv2D(400, (3,3), activation='relu'))
model.add(MaxPooling2D((2, 2)))
model.add(Conv2D(800, (3, 3), activation='relu'))
model.add(MaxPooling2D((2, 2)))
model.add(Conv2D(1000, (3,3), activation = 'relu'))
#model.add(Flatten())
model.add(GlobalAveragePooling2D())
model.add(Dense(400, activation = 'relu'))
model.add(Dense(200, activation = 'relu'))
model.add(Dense(2, activation = 'softmax'))
model.summary()

# compiling the CNN model

model.compile(optimizer = 'adamw', loss = 'sparse_categorical_crossentropy', metrics = ['accuracy'])

# Train the model

model.fit(x_train, y_train, epochs = 5, batch_size = 100, validation_data = (x_val, y_val))

#evaluating accuracy

y_pred = model.predict(x_val)
y_pred = tf.argmax(y_pred, axis = 1)
loss, acc = model.evaluate(x_val, y_val)
print ("validation accuracy:", acc)

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_12 (Conv2D)              │ (None, 298, 298, 400)  │        11,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_8 (MaxPooling2D)  │ (None, 149, 149, 400)  │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_13 (Conv2D)              │ (None, 147, 147, 800)  │     2,880,800 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_9 (MaxPooling2D)  │ (None, 73, 73, 800)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_14 (Conv2D)              │ (None, 71, 71, 1000)   │     7,201,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 1000)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 400)            │       400,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 200)            │        80,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 2)              │           402 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 10,574,002 (40.34 MB)

 Trainable params: 10,574,002 (40.34 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/5
